# LangChain Fundamentals: Part 1
## Building Your First LLM Application

### Learning Objectives
By the end of this notebook, you'll understand:
1. **What LangChain is** - A framework for building LLM applications
2. **Key Components** - LLMs, Prompts, Chains, Output Parsers
3. **The Pipe Operator** - How to connect components together
4. **Basic Operations** - invoke() for single inputs and batch() for multiple
5. **Output Parsing** - Extracting structured text from model responses

## What is LangChain?

**LangChain** is a framework that simplifies building applications with Large Language Models (LLMs).

Think of it like this:
- **Without LangChain**: Directly calling OpenAI's API is like assembling a bicycle from individual metal parts
- **With LangChain**: You get pre-made bicycle components that snap together easily

### Core Benefits:
- 📝 **Prompt Management** - Structured templates for creating prompts
- ⛓️ **Chaining** - Connect multiple steps together
- 🧠 **Memory** - Keep context across conversations
- 🔗 **Integration** - Connect to databases, APIs, and tools
- 📊 **Output Parsing** - Extract structured data from responses

## Setup: Loading Environment Variables

First, we'll set up the environment by loading your API keys (which should be in a `.env` file).

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=PendingDeprecationWarning)

from dotenv import load_dotenv, find_dotenv

# Load API keys from .env file
load_dotenv(find_dotenv())

print("✓ Environment variables loaded successfully!")

## Step 1: Initialize the LLM

We need to:
1. Create an LLM instance (we'll use OpenAI's GPT-4)
2. Set temperature to 0 for consistent, focused answers

**What's temperature?** 
- 0 = deterministic (same input → same output)
- 1 = creative and varied
- For learning, we use 0 to get predictable results

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Initialize the LLM (Language Model)
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0  # 0 = deterministic, focused answers
)

# Initialize embeddings (we'll use these later for semantic search)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("✓ LLM and embeddings initialized!")

## Step 2: Understanding Prompts

A **prompt** is what you send to the LLM. In LangChain, we use `PromptTemplate` to create reusable prompt structures.

### Why use templates?
Instead of hardcoding prompts, templates let you:
- Reuse the same structure with different inputs
- Keep prompts consistent
- Make code more maintainable

### Example Structure:
```
Question: {question}
Answer:
```
The `{question}` is a **placeholder** that will be filled in later.

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define a prompt template
template = """Question: {question}

Answer: """

prompt = PromptTemplate(
    template=template,
    input_variables=['question']  # The placeholders we'll fill in
)

# Define our first question
question = "What is LangChain?"

print("Prompt template created!")
print(f"\nWhen we ask: {question}")
print(f"\nThe final prompt will be:\n{prompt.format(question=question)}")

## Step 3: Direct LLM Invocation (Without Prompt Template)

Let's first see how to call the LLM directly with just a string. This shows the basic concept before adding more structure.

In [ ]:
# Call the LLM directly with a string
response = llm.invoke(question)

# The response is an AIMessage object
print("Response type:", type(response).__name__)
print("\nFull response object:")
print(response)
print("\n" + "="*50)
print("\nJust the text content:")
print(response.content)

## Step 4: Building Chains - The Pipe Operator

Now we'll build our first **chain** using the pipe operator `|`.

### What is a Chain?
A chain connects components so that:
- Output from one component → Input to the next component

### The Pipe Operator `|`
In LangChain, `|` connects components:
```python
chain = prompt | llm
```

This means:
1. Take the input
2. Format it with `prompt`
3. Send the formatted result to `llm`
4. Return the LLM's response

In [ ]:
# Create a chain: prompt → llm
chain = prompt | llm

print("✓ Chain created!")
print("\nChain structure: prompt | llm")
print(f"  1. Input: '{question}'")
print(f"  2. Formatted by prompt template")
print(f"  3. Sent to LLM")
print(f"  4. Returns AIMessage object")

## Step 5: Invoking the Chain

Now let's use our chain! The `invoke()` method runs the chain with a single input.

In [ ]:
# Run the chain with our question
response = chain.invoke(question)

print("Response from chain:")
print(response)

## Step 6: Extracting Just the Text

The response contains a lot of metadata. Often we just want the text content.
Use `.content` to extract just the text:

In [ ]:
# Extract just the text content
response = chain.invoke(question)
answer = response.content

print("Just the text answer:")
print(answer)

## Step 7: Batch Processing - Multiple Inputs at Once

Instead of calling the chain once, we can process multiple questions at once using `batch()`.

### Why batch?
- More efficient (parallel processing)
- Useful when you have many questions
- Keeps code concise

In [ ]:
# Create a list of questions
questions = [
    {'question': "What is LangChain?"},
    {'question': "Who can use LangChain?"},
    {'question': "What can I build with LangChain?"},
    {'question': "Is LangChain suitable for building Excel macros?"}
]

print(f"Processing {len(questions)} questions...\n")

# Process all questions
responses = chain.batch(questions)

print("Responses received!")
print(f"Type: {type(responses)}")
print(f"Number of responses: {len(responses)}")

## Step 8: Output Parsing - Cleaning Up Results

The LLM returns an `AIMessage` object with metadata. To get just the text, we use an **Output Parser**.

### Why use Output Parsers?
- Extract just what you need
- Can validate output format
- Can convert to specific types (string, JSON, list, etc.)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Create an output parser that returns just strings
parser = StrOutputParser()

# Update our chain: prompt → llm → parser
chain = prompt | llm | parser

print("✓ Updated chain with output parser!")
print("\nNew chain structure: prompt | llm | parser")
print("  Now the output is just text, not an AIMessage object")

## Step 9: Using the Improved Chain

Now with the output parser, `invoke()` returns just a string:

In [ ]:
# Invoke with a single question
answer = chain.invoke(question)

print("Type of response:", type(answer).__name__)  # Should be 'str' now
print("\nAnswer:")
print(answer)

## Step 10: Batch Processing with Output Parsing

Now batch() will also return strings instead of AIMessage objects:

In [ ]:
# Process all questions and get clean text output
answers = chain.batch(questions)

print(f"Processed {len(answers)} questions\n")

# Display the answers
for i, q in enumerate(questions, 1):
    print(f"Question {i}: {q['question']}")
    print(f"Answer: {answers[i-1][:150]}...")  # Show first 150 chars
    print("-" * 80)

## Summary: What We Learned

### Key Concepts:
1. **LLM** - The language model (GPT-4)
2. **PromptTemplate** - Structured prompts with placeholders
3. **Chains** - Connecting components with `|` operator
4. **invoke()** - Process a single input
5. **batch()** - Process multiple inputs efficiently
6. **Output Parser** - Extract clean text from responses

### Chain Evolution:
```python
# Simple chain
chain = prompt | llm

# Advanced chain with output parsing
chain = prompt | llm | parser
```

### Next Steps:
- In **Part 2**, we'll learn about **messages** for conversations
- How to build chatbots that remember context
- How to create more complex prompt templates

## Practice Exercises

Try these on your own:

### Exercise 1: Change the Temperature
What happens if you create a new LLM with `temperature=1`? How does the output differ?

### Exercise 2: Create Your Own Prompt Template
Create a new prompt template for a different use case (e.g., explaining concepts, writing stories, etc.)

### Exercise 3: Batch Processing
Create a list of 5 questions and process them with batch()